# 🛡️ PhishGuard Pro — Exploratory Data Analysis

This notebook provides a full EDA of the PhishingData.csv dataset used to train the four deep learning models.

**Sections:**
1. Dataset Overview
2. Label Distribution
3. Feature Statistics
4. Correlation Analysis
5. Feature-wise Distributions
6. Class Separability
7. Key Insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✔')

## 1. Dataset Overview

In [ ]:
df = pd.read_csv('../data/PhishingData.csv')
df.columns = df.columns.str.strip()

if 'index' in df.columns:
    df = df.drop(columns=['index'])

# Encode labels
df['Result'] = df['Result'].replace(-1, 0)

print(f'Shape: {df.shape}')
print(f'Features: {df.shape[1]-1}')
print(f'Samples: {df.shape[0]:,}')
df.head()

In [ ]:
print('Data Types:\n', df.dtypes.value_counts())
print('\nMissing Values:', df.isnull().sum().sum())
df.describe()

## 2. Label Distribution

In [ ]:
label_counts = df['Result'].value_counts()
label_names  = {0: 'Safe (Legitimate)', 1: 'Phishing'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar
axes[0].bar(
    [label_names[k] for k in label_counts.index],
    label_counts.values,
    color=['#22C55E', '#EF4444'],
    edgecolor='white', linewidth=1.5
)
axes[0].set_title('Sample Count by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)

# Pie
axes[1].pie(
    label_counts.values,
    labels=[label_names[k] for k in label_counts.index],
    autopct='%1.1f%%',
    colors=['#22C55E', '#EF4444'],
    startangle=90,
    explode=[0.05, 0.05]
)
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.suptitle('Label Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Feature Statistics

In [ ]:
X = df.drop(columns=['Result'])
y = df['Result']

feature_stats = pd.DataFrame({
    'Mean':   X.mean(),
    'Std':    X.std(),
    'Min':    X.min(),
    'Max':    X.max(),
    'Unique': X.nunique(),
})
print('Feature Statistics:')
feature_stats.style.background_gradient(cmap='Blues')

## 4. Correlation Analysis

In [ ]:
corr = df.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='RdYlGn', center=0,
    annot=False, linewidths=0.3,
    cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with target
target_corr = corr['Result'].drop('Result').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#EF4444' if v > 0 else '#22C55E' for v in target_corr.values]
target_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Target (Result)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

## 5. Feature-wise Distributions by Class

In [ ]:
top_features = target_corr.abs().sort_values(ascending=False).head(12).index.tolist()

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    for label, color, name in [(0, '#22C55E', 'Safe'), (1, '#EF4444', 'Phishing')]:
        data = df[df['Result'] == label][feat]
        ax.hist(data, bins=20, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('Value'); ax.set_ylabel('Count')

plt.suptitle('Top 12 Features — Class Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Class Separability (Violin Plots)

In [ ]:
top8 = target_corr.abs().sort_values(ascending=False).head(8).index.tolist()
df_melt = df[top8 + ['Result']].melt(id_vars='Result', var_name='Feature', value_name='Value')
df_melt['Class'] = df_melt['Result'].map({0: 'Safe', 1: 'Phishing'})

fig = px.violin(
    df_melt, x='Feature', y='Value', color='Class',
    box=True, points=False,
    color_discrete_map={'Safe': '#22C55E', 'Phishing': '#EF4444'},
    title='Class Separability — Top 8 Features'
)
fig.update_layout(height=500)
fig.show()

## 7. Key Insights

In [ ]:
print('='*60)
print('KEY DATASET INSIGHTS')
print('='*60)
print(f'Total samples         : {len(df):,}')
print(f'Phishing samples      : {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)')
print(f'Safe samples          : {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)')
print(f'Number of features    : {X.shape[1]}')
print(f'Missing values        : {df.isnull().sum().sum()}')
print(f'Feature value range   : [{X.min().min():.1f}, {X.max().max():.1f}]')
print()
print('Top 5 most predictive features:')
for i, (feat, corr_val) in enumerate(target_corr.abs().sort_values(ascending=False).head(5).items(), 1):
    direction = 'PHISHING' if target_corr[feat] > 0 else 'SAFE'
    print(f'  {i}. {feat:<35} corr={corr_val:.3f}  → {direction}')
print('='*60)